# Small LLM cerita Indonesia di Google Colab

Notebook ini mandiri: seluruh tokenisasi byte, Transformer, lima varian MLP, training, evaluasi, checkpoint, dan generasi ditulis di sini. Tidak ada impor dari repo lokal.

**Sebelum mulai:** buka [halaman dataset Indonesian Frog Storytelling Corpus](https://huggingface.co/datasets/bigscience-data/roots_id_indonesian_frog_storytelling_corpus), masuk ke akun Hugging Face, dan setujui syarat aksesnya. Dataset ini berlisensi CC BY-SA 4.0 dan mensyaratkan persetujuan BigScience Ethical Charter. Di Colab, pilih **Runtime → Change runtime type → GPU** bila tersedia, lalu jalankan sel dari atas ke bawah.

Korpus ini kecil dan berisi penceritaan lisan maupun tulisan atas rangkaian gambar yang sama. Loss validasi mengukur prediksi byte pada dokumen yang disisihkan; cerita yang terdengar runtut tetap perlu dinilai dengan membaca hasil generasi. Runtime Colab dapat terputus, jadi unduh checkpoint pada bagian akhir.

## 1. Pasang pembaca Parquet dan masuk ke Hugging Face

Sel login hanya meminta token jika belum ada sesi Hugging Face. Jangan tempel token langsung ke notebook.

In [ ]:
%pip -q install -U huggingface_hub pyarrow fsspec

In [ ]:
import gc
import json
import math
import random
import re
import time
import unicodedata
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from huggingface_hub import get_token, login
from torch import nn
from torch.nn import functional as F

if get_token() is None:
    login()

DATA_URL = "hf://datasets/bigscience-data/roots_id_indonesian_frog_storytelling_corpus/data/train-00000-of-00001.parquet"
try:
    df = pd.read_parquet(DATA_URL)
except Exception as exc:
    raise RuntimeError(
        "Gagal membaca dataset. Setujui akses di halaman Hugging Face, login dengan akun yang sama, lalu ulangi sel ini."
    ) from exc

print("Jumlah baris:", len(df))
print("Kolom:", list(df.columns))

## 2. Bersihkan teks dan pisahkan dokumen

Setiap baris teks diperlakukan sebagai satu dokumen. Baris kosong dan duplikat identik dibuang. Pemisahan train/validasi dilakukan **sebelum** dokumen digabung menjadi aliran token. Jika dataset tidak memiliki kolom `text`, lihat nama kolom yang dicetak di atas dan ubah `TEXT_COLUMN`.

In [ ]:
SEED = 42
TEXT_COLUMN = "text"
VALID_FRACTION = 0.20

if TEXT_COLUMN not in df.columns:
    raise ValueError(f"Kolom {TEXT_COLUMN!r} tidak ditemukan. Kolom tersedia: {list(df.columns)}")

def clean_text(value):
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")
    if not isinstance(value, str):
        return ""
    value = unicodedata.normalize("NFC", value)
    return re.sub(r"\s+", " ", value).strip()

documents = []
seen = set()
for value in df[TEXT_COLUMN]:
    story = clean_text(value)
    key = story.casefold()
    if story and key not in seen:
        seen.add(key)
        documents.append(story)

if len(documents) < 5:
    raise ValueError(f"Hanya ada {len(documents)} dokumen unik; perlu setidaknya 5.")

rng = random.Random(SEED)
rng.shuffle(documents)
valid_count = max(1, min(len(documents) - 2, round(len(documents) * VALID_FRACTION)))
valid_docs = documents[:valid_count]
train_docs = documents[valid_count:]
assert set(train_docs).isdisjoint(valid_docs)

END_OF_STORY = 256
VOCAB_SIZE = 257

def encode(text, end_of_story=False):
    ids = list(text.encode("utf-8"))
    return ids + [END_OF_STORY] if end_of_story else ids

def join_documents(items):
    ids = []
    for story in items:
        ids.extend(encode(story, end_of_story=True))
    return torch.tensor(ids, dtype=torch.long)

train_ids = join_documents(train_docs)
valid_ids = join_documents(valid_docs)
print(f"Dokumen train/valid: {len(train_docs)}/{len(valid_docs)}")
print(f"Byte train/valid: {len(train_ids):,}/{len(valid_ids):,}")
print("Contoh awal dokumen:", train_docs[0][:250])

## 3. Transformer kecil

Model menebak byte berikutnya. ID 0–255 mewakili byte UTF-8; ID 256 menandai akhir dokumen. Perhitungan loss menggunakan cross-entropy per byte.

In [ ]:
MODEL_CONFIG = {"context_length": 256, "n_embed": 192, "n_head": 4, "n_blocks": 4}

class DenseMLP(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.hidden = nn.Linear(width, 4 * width)
        self.proj = nn.Linear(4 * width, width)

    def forward(self, x):
        return self.proj(F.relu(self.hidden(x)))

class CausalAttention(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        if width % heads:
            raise ValueError("n_embed harus habis dibagi n_head")
        self.heads = heads
        self.head_width = width // heads
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.proj = nn.Linear(width, width)

    def forward(self, x):
        batch, length, width = x.shape
        q, k, v = self.qkv(x).split(width, dim=-1)
        q = q.reshape(batch, length, self.heads, self.head_width).transpose(1, 2)
        k = k.reshape(batch, length, self.heads, self.head_width).transpose(1, 2)
        v = v.reshape(batch, length, self.heads, self.head_width).transpose(1, 2)
        attended = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(attended.transpose(1, 2).reshape(batch, length, width))

class Block(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        self.ln_attention = nn.LayerNorm(width)
        self.attention = CausalAttention(width, heads)
        self.ln_mlp = nn.LayerNorm(width)
        self.mlp = DenseMLP(width)

    def forward(self, x):
        x = x + self.attention(self.ln_attention(x))
        return x + self.mlp(self.ln_mlp(x))

class StoryTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = dict(config)
        width = config["n_embed"]
        self.token_embedding = nn.Embedding(VOCAB_SIZE, width)
        self.position_embedding = nn.Embedding(config["context_length"], width)
        self.blocks = nn.ModuleList(
            Block(width, config["n_head"]) for _ in range(config["n_blocks"])
        )
        self.final_norm = nn.LayerNorm(width)
        self.lm_head = nn.Linear(width, VOCAB_SIZE)

    def forward(self, tokens, targets=None):
        batch, length = tokens.shape
        if length > self.config["context_length"]:
            raise ValueError("Urutan melebihi context_length")
        positions = torch.arange(length, device=tokens.device)
        x = self.token_embedding(tokens) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.final_norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
        return logits, loss

## 4. Varian MLP

Pilih `dense` untuk percobaan pertama. Varian `pruned`, `topk`, `moe`, dan `set` disediakan untuk perbandingan seperti pada proyek Small LLM. Perbandingan berjalan hanya jika sel opsional di bawah diaktifkan.

In [ ]:
VARIANTS = ("dense", "pruned", "topk", "moe", "set")
SPARSE_CONFIG = {
    "sparsity": 0.5, "active_fraction": 0.25, "n_experts": 4,
    "top_k": 2, "density": 0.5, "rewire_fraction": 0.1,
    "rewire_every": 50, "aux_coef": 0.01,
}

class PrunedMLP(nn.Module):
    def __init__(self, source, sparsity):
        super().__init__()
        self.hidden, self.proj = source.hidden, source.proj
        for name, weight in (("hidden_mask", self.hidden.weight), ("proj_mask", self.proj.weight)):
            mask = torch.ones(weight.numel(), dtype=torch.bool)
            count = int(weight.numel() * sparsity)
            if count:
                mask[weight.detach().abs().flatten().argsort()[:count]] = False
            self.register_buffer(name, mask.reshape_as(weight))

    def forward(self, x):
        hidden = F.relu(F.linear(x, self.hidden.weight * self.hidden_mask, self.hidden.bias))
        return F.linear(hidden, self.proj.weight * self.proj_mask, self.proj.bias)

class TopKMLP(nn.Module):
    def __init__(self, source, fraction):
        super().__init__()
        self.hidden, self.proj = source.hidden, source.proj
        self.k = max(1, round(self.hidden.out_features * fraction))

    def forward(self, x):
        activations = F.relu(self.hidden(x))
        if self.k == activations.shape[-1]:
            return self.proj(activations)
        values, indices = activations.topk(self.k, dim=-1)
        columns = F.embedding(indices, self.proj.weight.T.contiguous())
        return (values.unsqueeze(-1) * columns).sum(dim=-2) + self.proj.bias

class Expert(nn.Module):
    def __init__(self, width, hidden_width):
        super().__init__()
        self.hidden = nn.Linear(width, hidden_width)
        self.proj = nn.Linear(hidden_width, width)

    def forward(self, x):
        return self.proj(F.relu(self.hidden(x)))

class MoEMLP(nn.Module):
    def __init__(self, source, n_experts, top_k):
        super().__init__()
        if source.hidden.out_features % n_experts:
            raise ValueError("Lebar MLP harus habis dibagi jumlah expert")
        width = source.hidden.in_features
        expert_width = source.hidden.out_features // n_experts
        self.gate = nn.Linear(width, n_experts, bias=False)
        self.experts = nn.ModuleList(Expert(width, expert_width) for _ in range(n_experts))
        self.top_k = top_k
        self.aux_loss = None
        with torch.no_grad():
            for i, expert in enumerate(self.experts):
                start, end = i * expert_width, (i + 1) * expert_width
                expert.hidden.weight.copy_(source.hidden.weight[start:end])
                expert.hidden.bias.copy_(source.hidden.bias[start:end])
                expert.proj.weight.copy_(source.proj.weight[:, start:end])
                expert.proj.bias.copy_(source.proj.bias / n_experts)

    def forward(self, x):
        flat = x.reshape(-1, x.shape[-1])
        gate_logits = self.gate(flat)
        scores, chosen = gate_logits.topk(self.top_k, dim=-1)
        weights = scores.softmax(dim=-1)
        importance = gate_logits.softmax(dim=-1).mean(dim=0)
        load = F.one_hot(chosen, len(self.experts)).float().mean(dim=(0, 1))
        self.aux_loss = len(self.experts) * (importance * load).sum()
        output = torch.zeros_like(flat)
        for expert_id, expert in enumerate(self.experts):
            row, slot = torch.where(chosen == expert_id)
            if row.numel():
                contribution = expert(flat[row]) * (self.top_k * weights[row, slot, None])
                output = output.index_add(0, row, contribution)
        return output.reshape_as(x)

class SETMLP(nn.Module):
    def __init__(self, source, density):
        super().__init__()
        self.hidden, self.proj = source.hidden, source.proj
        for name, weight in (("hidden_mask", self.hidden.weight), ("proj_mask", self.proj.weight)):
            count = round(weight.numel() * density)
            mask = torch.zeros(weight.numel(), dtype=torch.bool)
            mask[torch.randperm(weight.numel())[:count]] = True
            self.register_buffer(name, mask.reshape_as(weight))
        self.enforce_masks()

    def masked_layers(self):
        yield self.hidden.weight, self.hidden_mask
        yield self.proj.weight, self.proj_mask

    @torch.no_grad()
    def enforce_masks(self):
        for weight, mask in self.masked_layers():
            weight.mul_(mask)

    def forward(self, x):
        hidden = F.relu(F.linear(x, self.hidden.weight * self.hidden_mask, self.hidden.bias))
        return F.linear(hidden, self.proj.weight * self.proj_mask, self.proj.bias)

def convert_model(model, variant, options):
    if variant not in VARIANTS:
        raise ValueError(f"Varian tidak dikenal: {variant}")
    for block in model.blocks:
        source = block.mlp
        if variant == "pruned":
            block.mlp = PrunedMLP(source, options["sparsity"])
        elif variant == "topk":
            block.mlp = TopKMLP(source, options["active_fraction"])
        elif variant == "moe":
            block.mlp = MoEMLP(source, options["n_experts"], options["top_k"])
        elif variant == "set":
            block.mlp = SETMLP(source, options["density"])
    return model

def moe_aux_loss(model):
    losses = [block.mlp.aux_loss for block in model.blocks if isinstance(block.mlp, MoEMLP)]
    return torch.stack(losses).sum() if losses else torch.zeros((), device=next(model.parameters()).device)

@torch.no_grad()
def rewire_set(model, optimizer, fraction):
    for block in model.blocks:
        if not isinstance(block.mlp, SETMLP):
            continue
        for weight, mask in block.mlp.masked_layers():
            flat_weight, flat_mask = weight.view(-1), mask.view(-1)
            active = flat_mask.nonzero(as_tuple=True)[0]
            dormant = (~flat_mask).nonzero(as_tuple=True)[0]
            count = min(round(active.numel() * fraction), dormant.numel())
            if count == 0:
                continue
            removed = active[flat_weight[active].abs().topk(count, largest=False).indices]
            grown = dormant[torch.randperm(dormant.numel(), device=dormant.device)[:count]]
            flat_mask[removed], flat_mask[grown] = False, True
            flat_weight[removed], flat_weight[grown] = 0, 0
            for value in optimizer.state.get(weight, {}).values():
                if isinstance(value, torch.Tensor) and value.shape == weight.shape:
                    value.view(-1)[removed] = 0
                    value.view(-1)[grown] = 0
        block.mlp.enforce_masks()

## 5. Pengaturan dan evaluasi

Ubah `steps` atau `batch_size` jika runtime terbatas. Evaluasi membaca seluruh byte validasi sekali per pemeriksaan, sehingga `valid_loss` stabil untuk pembagian data yang sama. `valid_byte_accuracy` adalah akurasi prediksi byte, bukan ukuran keruntutan cerita.

In [ ]:
TRAIN_CONFIG = {
    "batch_size": 4,
    "grad_accum": 2,
    "steps": 600,
    "eval_every": 50,
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "grad_clip": 1.0,
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_ROOT = Path("/content/small_llm_frog_runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if min(len(train_ids), len(valid_ids)) <= MODEL_CONFIG["context_length"]:
    raise ValueError("Data train atau validasi lebih pendek dari context_length.")
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("GPU Colab tidak aktif; training akan lebih lambat.")

def sample_batch(ids, generator):
    length = MODEL_CONFIG["context_length"]
    starts = torch.randint(len(ids) - length, (TRAIN_CONFIG["batch_size"],), generator=generator)
    positions = starts[:, None] + torch.arange(length)[None, :]
    return ids[positions].to(DEVICE), ids[positions + 1].to(DEVICE)

@torch.no_grad()
def evaluate_full(model):
    model.eval()
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    length = MODEL_CONFIG["context_length"]
    for start in range(0, len(valid_ids) - 1, length):
        end = min(start + length, len(valid_ids) - 1)
        x = valid_ids[start:end].unsqueeze(0).to(DEVICE)
        y = valid_ids[start + 1:end + 1].unsqueeze(0).to(DEVICE)
        logits, loss = model(x, y)
        count = y.numel()
        total_loss += loss.item() * count
        total_correct += (logits.argmax(dim=-1) == y).sum().item()
        total_tokens += count
    value = total_loss / total_tokens
    return {"valid_loss": value, "valid_byte_accuracy": total_correct / total_tokens,
            "valid_byte_perplexity": math.exp(value)}

## 6. Training dan penyimpanan checkpoint terbaik

Setiap percobaan memiliki folder baru. `best_model.pt` disimpan saat loss validasi membaik. Checkpoint berisi bobot untuk inferensi, bukan state optimizer untuk melanjutkan training.

In [ ]:
def train_variant(variant):
    if variant not in VARIANTS:
        raise ValueError(f"Varian tersedia: {VARIANTS}")
    random.seed(SEED)
    torch.manual_seed(SEED)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(SEED)
        torch.cuda.reset_peak_memory_stats()
    generator = torch.Generator().manual_seed(SEED + 1)
    model = convert_model(StoryTransformer(MODEL_CONFIG), variant, SPARSE_CONFIG).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=TRAIN_CONFIG["learning_rate"],
        weight_decay=TRAIN_CONFIG["weight_decay"],
    )
    run_dir = OUTPUT_ROOT / f"{variant}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
    run_dir.mkdir(parents=True)
    metrics_path = run_dir / "metrics.jsonl"
    checkpoint_path = run_dir / "best_model.pt"
    best_loss, best_step = float("inf"), None
    last_train_loss = None
    started = time.perf_counter()

    def record(step, train_loss, log):
        nonlocal best_loss, best_step
        scores = evaluate_full(model)
        row = {"step": step, "train_loss": train_loss, **scores}
        log.write(json.dumps(row) + "\n")
        log.flush()
        print(f"{variant} step {step:4d}: train={train_loss} valid={scores['valid_loss']:.4f} "
              f"accuracy={scores['valid_byte_accuracy']:.3f}")
        if scores["valid_loss"] < best_loss:
            best_loss, best_step = scores["valid_loss"], step
            torch.save({
                "model": model.state_dict(), "model_config": MODEL_CONFIG,
                "sparse_config": SPARSE_CONFIG, "variant": variant,
                "best_step": step, "best_valid_loss": best_loss,
            }, checkpoint_path)
        return scores

    with metrics_path.open("w", encoding="utf-8") as log:
        record(0, None, log)
        for step in range(1, TRAIN_CONFIG["steps"] + 1):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            train_loss = 0.0
            for _ in range(TRAIN_CONFIG["grad_accum"]):
                x, y = sample_batch(train_ids, generator)
                _, token_loss = model(x, y)
                loss = token_loss
                if variant == "moe":
                    loss = loss + SPARSE_CONFIG["aux_coef"] * moe_aux_loss(model)
                (loss / TRAIN_CONFIG["grad_accum"]).backward()
                train_loss += token_loss.item() / TRAIN_CONFIG["grad_accum"]
            if not math.isfinite(train_loss):
                raise FloatingPointError(f"Loss tidak finite pada step {step}")
            nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CONFIG["grad_clip"])
            optimizer.step()
            if variant == "set":
                for block in model.blocks:
                    block.mlp.enforce_masks()
                if step % SPARSE_CONFIG["rewire_every"] == 0:
                    rewire_set(model, optimizer, SPARSE_CONFIG["rewire_fraction"])
            last_train_loss = train_loss
            if step % TRAIN_CONFIG["eval_every"] == 0 or step == TRAIN_CONFIG["steps"]:
                record(step, train_loss, log)

    summary = {
        "variant": variant, "parameters": sum(p.numel() for p in model.parameters()),
        "train_documents": len(train_docs), "valid_documents": len(valid_docs),
        "train_bytes": len(train_ids), "valid_bytes": len(valid_ids),
        "best_step": best_step, "best_valid_loss": best_loss,
        "last_train_loss": last_train_loss,
        "seconds": time.perf_counter() - started,
        "device": DEVICE, "checkpoint_path": str(checkpoint_path),
        "metrics_path": str(metrics_path),
    }
    if DEVICE == "cuda":
        summary["peak_gpu_memory_mb"] = torch.cuda.max_memory_allocated() / 1_000_000
    (run_dir / "summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    del model, optimizer
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return summary

In [ ]:
VARIANT = "dense"
summary = train_variant(VARIANT)
display(pd.DataFrame([summary]).T.rename(columns={0: "hasil"}))

## 7. Lihat kurva loss dan hasil validasi

Kurva yang membaik menunjukkan model belajar menebak byte pada dokumen validasi. Jika loss train terus turun tetapi loss validasi naik, gunakan checkpoint dari `best_step`, bukan bobot langkah terakhir.

In [ ]:
metrics = pd.read_json(summary["metrics_path"], lines=True)
display(metrics)
ax = metrics.plot(x="step", y=["train_loss", "valid_loss"], marker="o", figsize=(8, 4))
ax.set_ylabel("Cross-entropy per byte")
ax.grid(True, alpha=0.3)
plt.show()

## 8. Coba generasi dari checkpoint terbaik

Karakter keluaran dibatasi pada karakter yang muncul di dokumen train agar urutan UTF-8 valid. Ini memperbaiki tampilan teks, bukan menjamin cerita masuk akal. Ubah prompt dan `temperature` untuk mencoba hasil lain.

In [ ]:
def build_character_trie(documents):
    root = {}
    for char in set("".join(documents)):
        if not char.isprintable() and char not in "\n\t":
            continue
        node = root
        for byte in char.encode("utf-8"):
            node = node.setdefault(byte, {})
    return root

CHAR_TRIE = build_character_trie(train_docs)

def bytes_to_finish(node):
    return 0 if not node else 1 + min(bytes_to_finish(child) for child in node.values())

@torch.no_grad()
def generate_text(model, prompt, max_new_bytes=200, temperature=0.7, top_k=40):
    if not prompt or temperature <= 0 or max_new_bytes < 1:
        raise ValueError("Prompt, temperature, dan max_new_bytes harus positif")
    model.eval()
    prefix = encode(prompt)
    generated = []
    node = CHAR_TRIE
    for step in range(max_new_bytes):
        context = (prefix + generated)[-MODEL_CONFIG["context_length"]:]
        x = torch.tensor([context], dtype=torch.long, device=DEVICE)
        logits, _ = model(x)
        scores = logits[0, -1].float() / temperature
        remaining = max_new_bytes - step
        allowed = [byte for byte, child in node.items()
                   if 1 + bytes_to_finish(child) <= remaining]
        if node is CHAR_TRIE:
            allowed.append(END_OF_STORY)
        masked = torch.full_like(scores, float("-inf"))
        masked[allowed] = scores[allowed]
        if top_k is not None:
            k = min(top_k, len(allowed))
            threshold = torch.topk(masked, k).values[-1]
            masked[masked < threshold] = float("-inf")
        next_byte = torch.multinomial(F.softmax(masked, dim=0), 1).item()
        if next_byte == END_OF_STORY:
            break
        generated.append(next_byte)
        node = node[next_byte]
        if not node:
            node = CHAR_TRIE
    return prompt + bytes(generated).decode("utf-8", errors="strict")

checkpoint = torch.load(summary["checkpoint_path"], map_location="cpu", weights_only=True)
best_model = convert_model(
    StoryTransformer(checkpoint["model_config"]),
    checkpoint["variant"], checkpoint["sparse_config"],
).to(DEVICE)
best_model.load_state_dict(checkpoint["model"])
print(generate_text(best_model, "Pada suatu pagi, "))

## 9. Opsional: bandingkan lima varian

Ubah `RUN_COMPARISON` menjadi `True` jika ingin melatih empat varian lain. Masing-masing memakai dokumen, seed, batch, dan jumlah langkah yang sama. Hasil yang sudah dilatih di atas dipakai kembali.

In [ ]:
RUN_COMPARISON = False
if RUN_COMPARISON:
    summaries = [summary]
    for variant in VARIANTS:
        if variant != summary["variant"]:
            summaries.append(train_variant(variant))
    comparison = pd.DataFrame(summaries).sort_values("best_valid_loss")
    display(comparison[["variant", "parameters", "best_step",
                        "best_valid_loss", "seconds", "checkpoint_path"]])
    comparison.to_csv(OUTPUT_ROOT / "comparison.csv", index=False)

## 10. Unduh hasil sebelum runtime Colab berhenti

Sel ini membuat ZIP berisi metrik, ringkasan, dan checkpoint yang sudah dihasilkan. Jika memakai semua varian, ZIP mencakup semuanya.

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive("/content/small_llm_frog_runs", "zip", OUTPUT_ROOT)
files.download(archive_path)